In [1]:
!pip install -q kagglehub

import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print("Downloaded to:", path)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Downloaded to: /kaggle/input/skin-cancer-mnist-ham10000


In [3]:
import os

csv_candidates = [os.path.join(root, f) for root, _, files in os.walk(path) for f in files if f.endswith(".csv")]
img_dir_candidates = [root for root, dirs, files in os.walk(path) if any(f.lower().endswith(".jpg") for f in files)]

print("CSV files found:", csv_candidates)
print("Image directories found:", img_dir_candidates)

CSV files found: ['/kaggle/input/skin-cancer-mnist-ham10000/hmnist_8_8_RGB.csv', '/kaggle/input/skin-cancer-mnist-ham10000/hmnist_28_28_RGB.csv', '/kaggle/input/skin-cancer-mnist-ham10000/hmnist_8_8_L.csv', '/kaggle/input/skin-cancer-mnist-ham10000/hmnist_28_28_L.csv', '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_metadata.csv']
Image directories found: ['/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_1', '/kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1', '/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2', '/kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_2']


In [4]:
import pandas as pd

metadata_path = next(p for p in csv_candidates if "metadata" in p.lower())
df = pd.read_csv(metadata_path)
print(df.shape)
print(df.columns.tolist())

class_counts = df["dx"].value_counts()
print()
print(class_counts)
print()
imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio (largest/smallest class): {imbalance_ratio:.1f}x")
majority_baseline_acc = class_counts.max() / len(df)
print(f"'Always predict {class_counts.idxmax()}' baseline accuracy: {majority_baseline_acc:.1%}")

(10015, 7)
['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']

dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

Imbalance ratio (largest/smallest class): 58.3x
'Always predict nv' baseline accuracy: 66.9%


In [5]:
def find_image_path(image_id):
    for img_dir in img_dir_candidates:
        candidate = os.path.join(img_dir, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None

df["image_path"] = df["image_id"].apply(find_image_path)

missing = df["image_path"].isna().sum()
print(f"Images not found: {missing} out of {len(df)}")
df = df.dropna(subset=["image_path"]).reset_index(drop=True)
print("Final shape:", df.shape)

Images not found: 0 out of 10015
Final shape: (10015, 8)


In [6]:
import numpy as np

DX_TO_FULLNAME = {
    "akiec": "Actinic keratoses and intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions",
}
DX_CODES = sorted(DX_TO_FULLNAME.keys())  # fixed order — this defines the model's output index order
CONDITIONS = [DX_TO_FULLNAME[c] for c in DX_CODES]

code_to_index = {code: i for i, code in enumerate(DX_CODES)}
df["label"] = df["dx"].map(code_to_index)

print("Class index mapping:")
for code, idx in code_to_index.items():
    print(f"  {idx}: {code} -> {DX_TO_FULLNAME[code]}")

Class index mapping:
  0: akiec -> Actinic keratoses and intraepithelial carcinoma
  1: bcc -> Basal cell carcinoma
  2: bkl -> Benign keratosis-like lesions
  3: df -> Dermatofibroma
  4: mel -> Melanoma
  5: nv -> Melanocytic nevi
  6: vasc -> Vascular lesions


In [7]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

Train: 8012, Val: 2003


In [8]:
import tensorflow as tf

IMG_SIZE = 224  # EfficientNetB0's standard input size

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    # NO /255 here — EfficientNet has rescaling built into its own first
    # layer and expects raw [0,255] pixel values. Normalizing here would
    # cause the model to double-rescale everything down to ~[0, 0.004],
    # silently wrecking training without any error.
    return image, label

def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=42)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df["image_path"].to_numpy(), train_df["label"].to_numpy(), shuffle=True)
val_ds = make_dataset(val_df["image_path"].to_numpy(), val_df["label"].to_numpy())

for imgs, labs in train_ds.take(1):
    print("batch image shape:", imgs.shape, "pixel range:", imgs.numpy().min(), "-", imgs.numpy().max())

batch image shape: (32, 224, 224, 3) pixel range: 0.0 - 255.0


In [9]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # phase 1: frozen base

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(CONDITIONS), activation="softmax")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 7)              │         8,967 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,058,538 (15.48 MB)

 Trainable params: 8,967 (35.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [10]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_df["label"])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["label"].to_numpy())
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))

print(f"{'Condition':<50} {'Weight':>8}")
for cls_idx, weight in sorted(class_weight_dict.items()):
    print(f"{CONDITIONS[cls_idx]:<50} {weight:>8.3f}")

Condition                                            Weight
Actinic keratoses and intraepithelial carcinoma       4.369
Basal cell carcinoma                                  2.785
Benign keratosis-like lesions                         1.302
Dermatofibroma                                       12.441
Melanoma                                              1.286
Melanocytic nevi                                      0.213
Vascular lesions                                     10.040


In [11]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase1.keras", save_best_only=True, monitor="val_loss"),
]

history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

Epoch 1/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 131s 402ms/step - accuracy: 0.4616 - loss: 1.5693 - val_accuracy: 0.5117 - val_loss: 1.3707
Epoch 2/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 49s 194ms/step - accuracy: 0.5640 - loss: 1.2360 - val_accuracy: 0.6350 - val_loss: 1.0580
Epoch 3/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 48s 191ms/step - accuracy: 0.6036 - loss: 1.1025 - val_accuracy: 0.6236 - val_loss: 1.0404
Epoch 4/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 48s 190ms/step - accuracy: 0.6103 - loss: 1.0519 - val_accuracy: 0.5871 - val_loss: 1.1217
Epoch 5/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 49s 194ms/step - accuracy: 0.6340 - loss: 0.9927 - val_accuracy: 0.6850 - val_loss: 0.9140
Epoch 6/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 46s 182ms/step - accuracy: 0.6296 - loss: 0.9540 - val_accuracy: 0.5916 - val_loss: 1.1124
Epoch 7/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 47s 187ms/step - accuracy: 0.6402 - loss: 0.9379 - val_accuracy: 0.6685 - val_loss: 0.9273
Epoch 8/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 47s 186ms/step - accuracy: 0.6574 - loss: 

In [12]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("checkpoint_phase2.keras", save_best_only=True, monitor="val_loss"),
]

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    shuffle=False,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

Epoch 1/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 95s 279ms/step - accuracy: 0.3682 - loss: 1.2665 - val_accuracy: 0.5107 - val_loss: 1.4260
Epoch 2/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 48s 193ms/step - accuracy: 0.4451 - loss: 1.1358 - val_accuracy: 0.5227 - val_loss: 1.3628
Epoch 3/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 48s 189ms/step - accuracy: 0.4973 - loss: 1.0628 - val_accuracy: 0.5552 - val_loss: 1.2652
Epoch 4/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 47s 186ms/step - accuracy: 0.5418 - loss: 1.0019 - val_accuracy: 0.5771 - val_loss: 1.1909
Epoch 5/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 83s 191ms/step - accuracy: 0.5642 - loss: 0.9542 - val_accuracy: 0.6026 - val_loss: 1.1332
Epoch 6/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 49s 194ms/step - accuracy: 0.5916 - loss: 0.9004 - val_accuracy: 0.6141 - val_loss: 1.0849
Epoch 7/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 48s 190ms/step - accuracy: 0.6111 - loss: 0.8833 - val_accuracy: 0.6296 - val_loss: 1.0485
Epoch 8/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 48s 192ms/step - accuracy: 0.6301 - loss: 0

In [13]:
from sklearn.metrics import classification_report, confusion_matrix

y_true = val_df["label"].to_numpy()
y_pred_probs = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

accuracy = (y_pred == y_true).mean()
print(f"v3 (class-weighted) accuracy: {accuracy:.1%}")
print(f"v2 (unweighted) accuracy:     80.5%")
print(f"Majority-class baseline:      {majority_baseline_acc:.1%}")
print()
print(classification_report(y_true, y_pred, target_names=CONDITIONS, zero_division=0))
print()
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

melanoma_idx = CONDITIONS.index("Melanoma")
melanoma_recall = confusion_matrix(y_true, y_pred)[melanoma_idx, melanoma_idx] / (y_true == melanoma_idx).sum()
print(f"\nMelanoma recall: {melanoma_recall:.2f} (v2 was 0.42, v1 was 0.35)")

v3 (class-weighted) accuracy: 69.2%
v2 (unweighted) accuracy:     80.5%
Majority-class baseline:      66.9%

                                                 precision    recall  f1-score   support

Actinic keratoses and intraepithelial carcinoma       0.44      0.77      0.56        65
                           Basal cell carcinoma       0.50      0.71      0.59       103
                  Benign keratosis-like lesions       0.55      0.55      0.55       220
                                 Dermatofibroma       0.19      0.70      0.30        23
                                       Melanoma       0.36      0.58      0.45       223
                               Melanocytic nevi       0.96      0.72      0.82      1341
                               Vascular lesions       0.38      0.89      0.54        28

                                       accuracy                           0.69      2003
                                      macro avg       0.48      0.70      0.54      2003

In [ ]:
import json, os

os.makedirs("model_artifacts", exist_ok=True)
model.save("model_artifacts/v3_weighted_efficientnet.keras")

with open("model_artifacts/condition_names.json", "w") as f:
    json.dump(CONDITIONS, f, indent=2)

print("Artifacts written:")
!ls -la model_artifacts

In [ ]:
import shutil
shutil.make_archive("v3_skin_lesion_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v3_skin_lesion_artifacts.zip")